In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported!")
print("🚀 Ready to predict India's future!")

✅ All libraries imported!
🚀 Ready to predict India's future!


In [3]:
# Load 2011 census data
census_2011 = pd.read_csv('../data/raw/india-districts-census-2011.csv')

# Calculate state-wise metrics for 2011
state_2011 = census_2011.groupby('State name').agg({
    'Population': 'sum',
    'Literate': 'sum',
    'Male': 'sum',
    'Female': 'sum'
}).reset_index()

state_2011['Literacy_Rate_2011'] = (state_2011['Literate'] / state_2011['Population']) * 100
state_2011['Sex_Ratio_2011'] = (state_2011['Female'] / state_2011['Male']) * 1000
state_2011['Population_Crore_2011'] = state_2011['Population'] / 10000000

state_2011 = state_2011[['State name', 'Population_Crore_2011', 'Literacy_Rate_2011', 'Sex_Ratio_2011']]
state_2011.columns = ['State', 'Population_2011', 'Literacy_2011', 'SexRatio_2011']

print("✅ 2011 data ready!")
print(state_2011.head())

✅ 2011 data ready!
                         State  Population_2011  Literacy_2011  SexRatio_2011
0  ANDAMAN AND NICOBAR ISLANDS         0.038058      77.324144     875.975374
1               ANDHRA PRADESH         8.458078      59.773345     992.848736
2            ARUNACHAL PRADESH         0.138373      55.358102     938.231883
3                        ASSAM         3.120558      61.456892     957.758248
4                        BIHAR        10.409945      50.436916     917.888480


In [4]:
# 2024 estimates (from various sources)
data_2024 = {
    'State': ['Uttar Pradesh', 'Maharashtra', 'Bihar', 'West Bengal', 
              'Madhya Pradesh', 'Tamil Nadu', 'Rajasthan', 'Karnataka',
              'Gujarat', 'Andhra Pradesh', 'Odisha', 'Telangana',
              'Kerala', 'Jharkhand', 'Assam', 'Punjab', 'Chhattisgarh',
              'Haryana', 'NCT OF DELHI', 'Jammu & Kashmir', 'Uttarakhand',
              'Himachal Pradesh', 'Tripura', 'Meghalaya', 'Manipur',
              'Nagaland', 'Goa', 'Arunachal Pradesh', 'Mizoram', 'Sikkim'],
    
    'Population_2024': [24.1, 12.5, 13.0, 10.2, 8.7, 7.7, 8.1, 6.8,
                        7.1, 5.4, 4.6, 3.9, 3.6, 4.0, 3.6, 3.1, 3.1,
                        3.0, 3.4, 1.4, 1.2, 0.7, 0.4, 0.3, 0.3,
                        0.2, 0.16, 0.16, 0.13, 0.07],
    
    'Literacy_2024': [73, 84, 74, 80, 73, 82, 75, 81, 84, 78,
                       81, 80, 96, 75, 86, 82, 78, 82, 90, 78,
                       88, 87, 92, 85, 88, 84, 92, 76, 95, 88],
    
    'SexRatio_2024': [995, 966, 1018, 973, 970, 1088, 952, 1034, 950,
                       1003, 1063, 1007, 1121, 1011, 1012, 938, 1015,
                       926, 870, 948, 1000, 1007, 980, 1013, 992, 931,
                       1009, 938, 1000, 938]
}

df_2024 = pd.DataFrame(data_2024)
print("✅ 2024 estimates ready!")
print(df_2024.head())

✅ 2024 estimates ready!
            State  Population_2024  Literacy_2024  SexRatio_2024
0   Uttar Pradesh             24.1             73            995
1     Maharashtra             12.5             84            966
2           Bihar             13.0             74           1018
3     West Bengal             10.2             80            973
4  Madhya Pradesh              8.7             73            970


In [5]:
# Merge 2011 and 2024 data
combined = pd.merge(state_2011, df_2024, on='State', how='inner')

print(f"✅ Merged data: {len(combined)} states")
print(combined.head())
print(f"\nColumns: {combined.columns.tolist()}")

✅ Merged data: 1 states
          State  Population_2011  Literacy_2011  SexRatio_2011  \
0  NCT OF DELHI         1.678794      75.874504     867.957277   

   Population_2024  Literacy_2024  SexRatio_2024  
0              3.4             90            870  

Columns: ['State', 'Population_2011', 'Literacy_2011', 'SexRatio_2011', 'Population_2024', 'Literacy_2024', 'SexRatio_2024']


In [6]:
# Calculate annual growth rates
years_gap = 2024 - 2011  # 13 years

# Population growth rate per year
combined['Pop_Growth_Rate'] = (
    ((combined['Population_2024'] / combined['Population_2011']) ** (1/years_gap)) - 1
) * 100

# Literacy improvement per year
combined['Literacy_Growth'] = (
    combined['Literacy_2024'] - combined['Literacy_2011']
) / years_gap

# Sex ratio change per year
combined['SexRatio_Change'] = (
    combined['SexRatio_2024'] - combined['SexRatio_2011']
) / years_gap

print("📊 GROWTH RATES (Per Year):\n")
print(combined[['State', 'Pop_Growth_Rate', 'Literacy_Growth', 'SexRatio_Change']].round(2).head(10))

📊 GROWTH RATES (Per Year):

          State  Pop_Growth_Rate  Literacy_Growth  SexRatio_Change
0  NCT OF DELHI             5.58             1.09             0.16


In [7]:
# Predict 2030 values
years_to_predict = 2030 - 2024  # 6 years from now

# Population prediction (compound growth)
combined['Population_2030'] = combined['Population_2024'] * (
    (1 + combined['Pop_Growth_Rate']/100) ** years_to_predict
)

# Literacy prediction (linear improvement, max 100%)
combined['Literacy_2030'] = combined['Literacy_2024'] + (
    combined['Literacy_Growth'] * years_to_predict
)
combined['Literacy_2030'] = combined['Literacy_2030'].clip(upper=100)

# Sex ratio prediction (linear change)
combined['SexRatio_2030'] = combined['SexRatio_2024'] + (
    combined['SexRatio_Change'] * years_to_predict
)

print("🔮 INDIA 2030 PREDICTIONS:\n")
print(combined[['State', 'Population_2030', 'Literacy_2030', 'SexRatio_2030']].round(2).head(10))

🔮 INDIA 2030 PREDICTIONS:

          State  Population_2030  Literacy_2030  SexRatio_2030
0  NCT OF DELHI             4.71          96.52         870.94


In [8]:
print("="*60)
print("🇮🇳 INDIA 2030 PREDICTIONS - KEY FINDINGS")
print("="*60)

# Total population prediction
total_pop_2030 = combined['Population_2030'].sum()
total_pop_2024 = combined['Population_2024'].sum()
print(f"\n📊 INDIA TOTAL POPULATION:")
print(f"   2024: {total_pop_2024:.1f} Crore")
print(f"   2030: {total_pop_2030:.1f} Crore")
print(f"   Growth: +{(total_pop_2030 - total_pop_2024):.1f} Crore")

# Most populated state in 2030
top_pop = combined.nlargest(1, 'Population_2030').iloc[0]
print(f"\n🏆 MOST POPULATED STATE IN 2030:")
print(f"   {top_pop['State']}: {top_pop['Population_2030']:.1f} Crore")

# Highest literacy in 2030
top_lit = combined.nlargest(1, 'Literacy_2030').iloc[0]
print(f"\n📚 MOST LITERATE STATE IN 2030:")
print(f"   {top_lit['State']}: {top_lit['Literacy_2030']:.1f}%")

# States reaching 100% literacy
fully_literate = combined[combined['Literacy_2030'] >= 99]
print(f"\n🎓 STATES NEAR 100% LITERACY BY 2030:")
for _, row in fully_literate.iterrows():
    print(f"   ✅ {row['State']}: {row['Literacy_2030']:.1f}%")

# Best sex ratio prediction
best_sr = combined.nlargest(1, 'SexRatio_2030').iloc[0]
print(f"\n👫 BEST SEX RATIO IN 2030:")
print(f"   {best_sr['State']}: {best_sr['SexRatio_2030']:.0f} females per 1000 males")

🇮🇳 INDIA 2030 PREDICTIONS - KEY FINDINGS

📊 INDIA TOTAL POPULATION:
   2024: 3.4 Crore
   2030: 4.7 Crore
   Growth: +1.3 Crore

🏆 MOST POPULATED STATE IN 2030:
   NCT OF DELHI: 4.7 Crore

📚 MOST LITERATE STATE IN 2030:
   NCT OF DELHI: 96.5%

🎓 STATES NEAR 100% LITERACY BY 2030:

👫 BEST SEX RATIO IN 2030:
   NCT OF DELHI: 871 females per 1000 males


In [9]:
# Top 10 states by predicted 2030 population
top_10_pop = combined.nlargest(10, 'Population_2030')

fig = go.Figure()

fig.add_trace(go.Bar(
    name='2011',
    x=top_10_pop['State'],
    y=top_10_pop['Population_2011'],
    marker_color='lightblue'
))

fig.add_trace(go.Bar(
    name='2024',
    x=top_10_pop['State'],
    y=top_10_pop['Population_2024'],
    marker_color='steelblue'
))

fig.add_trace(go.Bar(
    name='2030 (Predicted)',
    x=top_10_pop['State'],
    y=top_10_pop['Population_2030'],
    marker_color='darkred'
))

fig.update_layout(
    title='📈 Population Growth: 2011 → 2024 → 2030 (Predicted)',
    xaxis_title='State',
    yaxis_title='Population (Crore)',
    barmode='group',
    height=600,
    xaxis_tickangle=-45
)

fig.show()
print("✅ Population prediction chart ready!")

✅ Population prediction chart ready!


In [11]:
# Literacy progression
fig2 = go.Figure()

# Sort by 2030 literacy
sorted_data = combined.sort_values('Literacy_2030', ascending=False).head(15)

fig2.add_trace(go.Scatter(
    name='2011',
    x=sorted_data['State'],
    y=sorted_data['Literacy_2011'],
    mode='markers+lines',
    marker=dict(size=10, color='lightcoral'),
    line=dict(width=2)
))

fig2.add_trace(go.Scatter(
    name='2024',
    x=sorted_data['State'],
    y=sorted_data['Literacy_2024'],
    mode='markers+lines',
    marker=dict(size=10, color='orange'),
    line=dict(width=2)
))

fig2.add_trace(go.Scatter(
    name='2030 (Predicted)',
    x=sorted_data['State'],
    y=sorted_data['Literacy_2030'],
    mode='markers+lines',
    marker=dict(size=12, color='green'),
    line=dict(width=3)
))

fig2.update_layout(
    title='📚 Literacy Rate Journey: 2011 → 2024 → 2030 (Predicted)',
    xaxis_title='State',
    yaxis_title='Literacy Rate (%)',
    height=600,
    xaxis_tickangle=-45,
    hovermode='x unified'
)

fig2.show()
print("✅ Literacy prediction chart ready!")

✅ Literacy prediction chart ready!


In [12]:
# Save all predictions
combined.to_csv('../data/cleaned/india_2030_predictions.csv', index=False)

print("✅ Predictions saved!")
print(f"📁 File: data/cleaned/india_2030_predictions.csv")
print(f"📊 Total states predicted: {len(combined)}")
print(f"\n🎯 Ready to add to Streamlit dashboard!")

✅ Predictions saved!
📁 File: data/cleaned/india_2030_predictions.csv
📊 Total states predicted: 1

🎯 Ready to add to Streamlit dashboard!
